
# 🏠 Predict How Fast Your Home Would Sell — End‑to‑End (Colab)
This notebook mirrors the **structure of your original “Loan Eligibility” template** but adapted to the housing project.  
It includes: steps/markdown cells, data load, preprocessing, model build/save, Streamlit frontend, and ngrok launcher.



## Steps
1. Loading the dataset  
2. Pre‑processing the dataset  
3. Building and saving the prediction model  
4. Building the Streamlit frontend and deploying


### 0) (Optional) Mount Google Drive

In [17]:
# Retrieve csv file from google drive by mapping the folder from google drive. Must be done each time session expires.
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [18]:

%cd /content/drive/MyDrive/Colab_Notebooks/Personal Project

[WinError 3] The system cannot find the path specified: '/content/drive/MyDrive/Colab_Notebooks/Personal Project'
c:\Users\terah\Downloads


### 1) Loading the dataset

In [19]:

import pandas as pd

# Path to CSV (change this if your file is on Drive)


df =pd.read_csv("C:\\Users\\terah\\Downloads\\housing_quick_sale.csv")
df.head()


,size_sqft,bedrooms,bathrooms,levels,distance_to_city_km,has_pool,basement_finished_sqft,basement_unfinished_sqft,lot_size_sqft,land_size_acres,garage_capacity,sold_quickly
0,2969,4,3.4,3,17.8,0,236,975,2000,0.101,2,1
1,2877,3,3.1,3,2.4,1,0,0,5074,0.229,2,1
2,1858,2,1.4,2,28.1,0,367,232,4318,0.122,2,0
3,2259,3,2.5,3,14.4,0,639,149,4388,0.103,2,1
4,2833,4,2.7,1,11.7,0,586,786,4453,0.248,2,1


### 2) Pre‑processing the dataset

In [20]:

# Select ONLY the major predictors requested
FEATURES = ["size_sqft","bedrooms","bathrooms","has_pool","land_size_acres"]
TARGET = "sold_quickly"

X = df[FEATURES].copy()
y = df[TARGET].astype(int)

X.head(), y.head()


(   size_sqft  bedrooms  bathrooms  has_pool  land_size_acres
 0       2969         4        3.4         0            0.101
 1       2877         3        3.1         1            0.229
 2       1858         2        1.4         0            0.122
 3       2259         3        2.5         0            0.103
 4       2833         4        2.7         0            0.248,
 0    1
 1    1
 2    0
 3    1
 4    1
 Name: sold_quickly, dtype: int64)

### 3) Building the prediction model

In [21]:

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report
import joblib
from pathlib import Path

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Use a higher max_iter to avoid convergence warnings
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_tr, y_tr)

probs = model.predict_proba(X_te)[:,1]
auc = roc_auc_score(y_te, probs)
print("Validation ROC AUC:", round(auc, 3))
print(classification_report(y_te, (probs>=0.5).astype(int)))

# Save model + features together
Path("models").mkdir(exist_ok=True, parents=True)
joblib.dump({"model": model, "features": FEATURES}, "models/classifier.pkl")
print("Saved → models/classifier.pkl")


Validation ROC AUC: 0.635
              precision    recall  f1-score   support

           0       0.51      0.65      0.58       135
           1       0.64      0.50      0.56       165

    accuracy                           0.57       300
   macro avg       0.58      0.57      0.57       300
weighted avg       0.58      0.57      0.57       300

Saved → models/classifier.pkl


### 4) Building the Frontend of the Application and Deploying the Model

#### 4.1 Install Required Libraries

In [22]:

!pip -q install streamlit pyngrok


In [23]:
!pip -q install pyngrok streamlit

In [24]:
!pip -q install streamlit cloudflared

#### 4.2 Create the frontend with Streamlit

In [25]:
%%writefile app.py
# app.py — Streamlit frontend for housing quick-sale prediction
import streamlit as st
import pandas as pd
import joblib
from pathlib import Path

st.set_page_config(page_title="🏠 Will My Home Sell Quickly?", layout="centered")

BUNDLE_PATH = Path("models/classifier.pkl")
if not BUNDLE_PATH.exists():
    st.error("Model file not found. Run the training cells above.")
    st.stop()

bundle = joblib.load(BUNDLE_PATH)
model = bundle["model"]
FEATURES = bundle["features"]

st.markdown(
    """
    <div style="background-color:yellow;padding:13px">
      <h1 style="color:black;text-align:center;">Predict How Fast Your Home Would Sell</h1>
    </div>
    """,
    unsafe_allow_html=True
)

col1, col2 = st.columns(2)
with col1:
    size_sqft = st.number_input("Home Size (sqft)", min_value=300, max_value=10000, value=2200, step=50)
    bedrooms = st.selectbox("Bedrooms", [1,2,3,4,5,6], index=2)
with col2:
    bathrooms = st.selectbox("Bathrooms", [1.0,1.5,2.0,2.5,3.0,3.5,4.0], index=2)
    has_pool_txt = st.radio("Pool?", ["No","Yes"], index=0)

land_size_acres = st.number_input(
    "Land Size (acres)",
    min_value=0.01,
    max_value=5.0,
    value=0.20,
    step=0.01,
    format="%.2f"
)

if st.button("Check"):
    row = pd.DataFrame([{
        "size_sqft": int(size_sqft),
        "bedrooms": int(bedrooms),
        "bathrooms": float(bathrooms),
        "has_pool": 1 if has_pool_txt == "Yes" else 0,
        "land_size_acres": float(land_size_acres)
    }])[FEATURES]

    proba = float(model.predict_proba(row)[0, 1])
    label = "Will Sell Quickly (≤ 30 days)" if proba >= 0.5 else "May Take Longer"

    st.metric("Probability (Sell Quickly)", f"{proba:.2%}")
    if proba >= 0.5:
        st.success(f"Prediction: {label}")
    else:
        st.warning(f"Prediction: {label}")


Overwriting app.py


#### 4.3 Launch in Colab with ngrok

In [ ]:
# running the app

!streamlit run app.py 

#!streamlit run app.py &>/dev/null&


---
### Troubleshooting
- If you see **convergence warnings**, we already set `max_iter=1000`. You can also try `solver="liblinear"`.
- If you see **feature name warnings**, ensure `row[FEATURES]` is used and that the model bundle includes the same `FEATURES` list.
- Re-run ngrok cell if the URL expires.
